In [1]:
import pandas as pd
import numpy as np
import openpyxl
import locale
import sys
sys.path.append('../')
from src import sp_eda as se
pd.set_option('display.max_columns', None)

Insertamos el dataframe y lo copiamos

In [2]:
df_raw=pd.read_csv('../data/Acc_met.csv')

df=df_raw.copy()
df.sample(5)

,State,Date,acc_events,acc_sev_1,acc_sev_2,acc_sev_3,acc_sev_4,weather_events,typ_Cold,typ_Fog,typ_Hail,typ_Precipitation,typ_Rain,typ_Snow,typ_Storm,wea_sev_Heavy,wea_sev_Light,wea_sev_Moderate,wea_sev_Other,wea_sev_Severe,wea_sev_UNK,Name
4533,IL,2022-08-08,113.0,22.0,62.0,25.0,4.0,146.0,2.0,16.0,0.0,16.0,112.0,0.0,0.0,6.0,94.0,15.0,0.0,15.0,16.0,Illinois
3022,FL,2022-06-11,253.0,0.0,240.0,10.0,3.0,250.0,13.0,15.0,0.0,34.0,187.0,0.0,1.0,23.0,137.0,34.0,0.0,22.0,34.0,Florida
1654,CO,2022-07-21,56.0,4.0,40.0,7.0,5.0,45.0,3.0,7.0,0.0,1.0,33.0,0.0,1.0,0.0,32.0,6.0,0.0,6.0,1.0,Colorado
11792,NY,2022-09-08,225.0,2.0,213.0,8.0,2.0,99.0,2.0,61.0,0.0,0.0,36.0,0.0,0.0,1.0,34.0,3.0,0.0,61.0,0.0,New York
15152,UT,2022-01-20,40.0,0.0,38.0,2.0,0.0,38.0,0.0,36.0,0.0,0.0,2.0,0.0,0.0,0.0,2.0,14.0,0.0,22.0,0.0,Utah


Movemos la columna de posición y cambiamos de nombre la columna con el codigo de estado.

In [3]:
col = "Name"
cols = list(df.columns)
cols.insert(1, cols.pop(cols.index(col)))
df= df[cols]
df = df.rename(columns={
    "State": "State_code",
    "Name": "State"
})



df.sample(5)

,State_code,State,Date,acc_events,acc_sev_1,acc_sev_2,acc_sev_3,acc_sev_4,weather_events,typ_Cold,typ_Fog,typ_Hail,typ_Precipitation,typ_Rain,typ_Snow,typ_Storm,wea_sev_Heavy,wea_sev_Light,wea_sev_Moderate,wea_sev_Other,wea_sev_Severe,wea_sev_UNK
17341,WY,Wyoming,2022-02-20,NaN,NaN,NaN,NaN,NaN,11.0,0.0,0.0,0.0,0.0,7.0,1.0,3.0,0.0,8.0,0.0,0.0,3.0,0.0
15622,VA,Virginia,2022-05-11,207.0,21.0,160.0,19.0,7.0,43.0,4.0,10.0,0.0,0.0,29.0,0.0,0.0,0.0,29.0,5.0,0.0,9.0,0.0
8737,MT,Montana,2022-03-14,16.0,0.0,16.0,0.0,0.0,27.0,1.0,18.0,0.0,0.0,0.0,7.0,1.0,0.0,7.0,2.0,0.0,18.0,0.0
3924,IA,Iowa,2022-12-02,4.0,0.0,4.0,0.0,0.0,9.0,1.0,1.0,0.0,0.0,2.0,4.0,1.0,0.0,6.0,1.0,0.0,2.0,0.0
9141,NC,North Carolina,2022-04-23,182.0,0.0,166.0,4.0,12.0,45.0,1.0,42.0,0.0,0.0,2.0,0.0,0.0,0.0,2.0,7.0,0.0,36.0,0.0


In [4]:
se.eda_prelim(df)

----------
DIMENSIONES
El conjunto de datos presenta 17653 filas y 22 columnas
----------
INFORMACION DE COLUMNAS
<class 'pandas.DataFrame'>
RangeIndex: 17653 entries, 0 to 17652
Data columns (total 22 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   State_code         17653 non-null  str    
 1   State              17653 non-null  str    
 2   Date               17653 non-null  str    
 3   acc_events         14431 non-null  float64
 4   acc_sev_1          14431 non-null  float64
 5   acc_sev_2          14431 non-null  float64
 6   acc_sev_3          14431 non-null  float64
 7   acc_sev_4          14431 non-null  float64
 8   weather_events     16714 non-null  float64
 9   typ_Cold           16714 non-null  float64
 10  typ_Fog            16714 non-null  float64
 11  typ_Hail           16714 non-null  float64
 12  typ_Precipitation  16714 non-null  float64
 13  typ_Rain           16714 non-null  float64
 14  typ_Snow       

Se van a unificar el tipo de datos de las columnas y vamos a usar tipo int, ya que para contar eventos nos es más que suficiente y es más eficiente.
En las filas de severidad de eventos meteorologicos, vamos a sustituir NaN por 0.

In [6]:

cols = df.columns[3:21]

df[cols] = df[cols].fillna(0).astype("int64")


Por otro lado, en la clasificación de eventos meteorologicos, existen dos tipos que vamos a unificar, ya que la información que aportan por separado no aporta más que toda ella junta (other y unknown).


In [7]:
df["wea_sev_UNKNOWN"] = (
    df[["wea_sev_UNK", "wea_sev_Other"]]
    .fillna(0)
    .sum(axis=1)
    .astype("Int64")
)

df = df.drop(columns=["wea_sev_UNK", "wea_sev_Other"])

Ponemos los nombres de las columnas todas en mayúsculas por estandarizar

In [8]:
df.columns = df.columns.str.upper()


Ordeno las columnas por severidad de evento meteorologico

In [9]:

old_pos = 16   # columna 16
new_pos = 18   # columna 18

cols = list(df.columns)
col = cols.pop(old_pos)
cols.insert(new_pos, col)
df = df[cols]


Vamos a cambiar el nombre de la severidad de accidentes, de 1, 2 3 y 4 a LIGHT, MODERATE, HEAVY y SEVERE respectivamente

In [10]:

cols_to_change = df.columns[4:8]

replacements = {
    "1": "LIGHT",
    "2": "MODERATE",
    "3": "HEAVY",
    "4": "SEVERE"
}
new_names = [
    col.replace("1", "LIGHT")
       .replace("2", "MODERATE")
       .replace("3", "HEAVY")
       .replace("4", "SEVERE")
    for col in cols_to_change
]
df.rename(columns=dict(zip(cols_to_change, new_names)), inplace=True)


Con esto damos por concluida la limpieza de datos y procedemos a guardar nuestro df 

In [13]:
df.to_csv('../data/Acc_met_clean.csv', index=False)

In [12]:
se.eda_prelim(df)
df.sample(5)

----------
DIMENSIONES
El conjunto de datos presenta 17653 filas y 21 columnas
----------
INFORMACION DE COLUMNAS
<class 'pandas.DataFrame'>
RangeIndex: 17653 entries, 0 to 17652
Data columns (total 21 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   STATE_CODE         17653 non-null  str  
 1   STATE              17653 non-null  str  
 2   DATE               17653 non-null  str  
 3   ACC_EVENTS         17653 non-null  int64
 4   ACC_SEV_LIGHT      17653 non-null  int64
 5   ACC_SEV_MODERATE   17653 non-null  int64
 6   ACC_SEV_HEAVY      17653 non-null  int64
 7   ACC_SEV_SEVERE     17653 non-null  int64
 8   WEATHER_EVENTS     17653 non-null  int64
 9   TYP_COLD           17653 non-null  int64
 10  TYP_FOG            17653 non-null  int64
 11  TYP_HAIL           17653 non-null  int64
 12  TYP_PRECIPITATION  17653 non-null  int64
 13  TYP_RAIN           17653 non-null  int64
 14  TYP_SNOW           17653 non-null  int64
 15  T

,STATE_CODE,STATE,DATE,ACC_EVENTS,ACC_SEV_LIGHT,ACC_SEV_MODERATE,ACC_SEV_HEAVY,ACC_SEV_SEVERE,WEATHER_EVENTS,TYP_COLD,TYP_FOG,TYP_HAIL,TYP_PRECIPITATION,TYP_RAIN,TYP_SNOW,TYP_STORM,WEA_SEV_LIGHT,WEA_SEV_MODERATE,WEA_SEV_HEAVY,WEA_SEV_SEVERE,WEA_SEV_UNKNOWN
14616,TN,Tennessee,2022-08-01,87,8,69,9,1,64,0,14,0,5,45,0,0,31,12,2,14,5
3384,GA,Georgia,2022-06-08,65,2,39,16,8,135,0,25,0,6,104,0,0,95,6,4,24,6
7760,MN,Minnesota,2022-07-08,59,0,58,0,1,172,4,11,0,6,151,0,0,122,26,7,11,6
2027,CT,Connecticut,2022-07-30,43,0,34,9,0,12,1,0,0,0,11,0,0,10,1,0,1,0
11182,NM,New Mexico,2022-12-28,1,0,1,0,0,58,0,4,0,0,29,12,13,41,4,0,13,0
